# Chapter 7: Serving and Deployment

*Small Language Models in Practice — Haji Gul*

> Turning a notebook model into a service others can call; a minimal but production-
shaped FastAPI endpoint with streaming; serving through Ollama's built-in API;
and the practical knobs — concurrency, batching, and health checks.

---

*Lecture notes mirroring the book. Run the setup cell, then work top-to-bottom. Swap model ids freely.*

## Setup
Uncomment what this chapter needs.

In [ ]:
# %pip install -q transformers datasets accelerate torch
# Chapter-specific installs appear in shell cells below.

## From script to service

Everything so far ran in a notebook. To put a model behind a product you need an
**HTTP endpoint**: a long-running process that loads the model once and
answers many requests. We will build one with FastAPI, then show the
zero-code alternative via Ollama.

## A FastAPI generation service

In [ ]:
%%bash
pip install fastapi uvicorn

Load the model *once* at startup, not per request — loading is the slow
part.

In [ ]:
# file: server.py
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

app = FastAPI(title="Local SLM API")

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)

class Query(BaseModel):
    prompt: str
    max_new_tokens: int = 200

@app.get("/health")          # cheap check for load balancers
def health():
    return {"status": "ok", "model": MODEL_ID}

@app.post("/generate")
def generate(q: Query):
    msgs = [{"role": "user", "content": q.prompt}]
    ids = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(ids, max_new_tokens=q.max_new_tokens)
    text = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
    return {"response": text}

Run it and call it:

In [ ]:
%%bash
uvicorn server:app --host 0.0.0.0 --port 8000

# In another terminal:
curl -X POST http://localhost:8000/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt": "Define quantization in one line."}'

> **Load once, serve many.** The model is loaded at import time, so every request reuses it. Never load inside
the request handler — that would re-read gigabytes of weights on every call and
make the service unusably slow.

## Streaming responses

For chat UIs, stream tokens instead of waiting for the full answer. FastAPI's
`StreamingResponse` plus a threaded generation streamer does it.

In [ ]:
from fastapi.responses import StreamingResponse
from transformers import TextIteratorStreamer
from threading import Thread

@app.post("/stream")
def stream(q: Query):
    msgs = [{"role": "user", "content": q.prompt}]
    ids = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(inputs=ids, max_new_tokens=q.max_new_tokens, streamer=streamer)
    Thread(target=model.generate, kwargs=kwargs).start()

    def token_iter():
        for token in streamer:
            yield token
    return StreamingResponse(token_iter(), media_type="text/plain")

## The no-code path: Ollama serve

If you do not need custom endpoints, Ollama already *is* a server with an
OpenAI-compatible API. Start it once and any OpenAI client library can talk to
it by pointing at the local URL.

In [ ]:
%%bash
ollama serve            # exposes http://localhost:11434

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
resp = client.chat.completions.create(
    model="qwen2.5:0.5b",
    messages=[{"role": "user", "content": "Hello from the OpenAI client!"}],
)
print(resp.choices[0].message.content)

> **Tip.** For higher throughput with many concurrent users on a GPU, look at
**vLLM**, which adds continuous batching and a drop-in OpenAI-compatible
server (`vllm serve <model>`). FastAPI is perfect for single-model,
moderate-traffic local services; vLLM is for scale.

## Recap and exercise

You wrapped a model in a FastAPI service with health, generation, and streaming
endpoints, and saw the zero-code Ollama alternative.

**Exercise.** Add a `/summarize` endpoint that accepts long text
and returns a three-sentence summary by composing a fixed instruction with the
input. Test it with `curl`.